# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset URL:**  
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.


In [ ]:
# List all record sets and their fields/columns by '@id'
print("Available Record Sets:")
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet: {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    Field: {field['@id']} | Name: {field.get('name', '')}")
    if 'columns' in rs:
        for col in rs['columns']:
            print(f"    Column: {col['@id']} | Name: {col.get('name', '')}")
    print()

print("\nSample record for each RecordSet:")
for rs in dataset.metadata.record_sets:
    rs_id = rs['@id']
    try:
        records_iter = dataset.records(record_set=rs_id)
        sample = next(records_iter)
        print(f"✔ {rs_id}:\n  Sample: {sample}\n")
    except Exception as e:
        print(f"✗ {rs_id} - Could not read sample: {e}\n")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s found above.


In [ ]:
# Extract all record set @id from metadata
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print(f"All record sets: {record_sets}")

# Load all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}.")
(print(f"Fields for {rs_id}: {df.columns.tolist()}\n") for rs_id, df in dataframes.items());

# As an example, select the first Record Set loaded
example_record_set_id = record_sets[0] if len(record_sets) else None
if example_record_set_id:
    print(f"Showing fields of {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data.


In [ ]:
# Example: Numeric filtering, normalization, and grouping
from pandas.api.types import is_numeric_dtype

# We'll work with the example Record Set (edit as needed)
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Find numeric columns
numeric_columns = [c for c in df.columns if is_numeric_dtype(df[c])]
print(f"Numeric fields available in {record_set_id}:", numeric_columns)

if numeric_columns:
    numeric_field = numeric_columns[0]  # pick the first numeric field
    threshold = df[numeric_field].mean()  # Example threshold: mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / 
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical field
    group_field = None
    # Try to find a non-numeric field with few unique values
    for col in df.columns:
        if col not in numeric_columns and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped (mean) {numeric_field} by {group_field}:")
        display(grouped_df)
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric fields found in this record set. EDA cannot proceed.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Example: Plot distribution of the selected numeric field after normalization
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True)
    plt.title(f"Distribution of Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Scatter plot against another numeric field, if available
    if len(numeric_columns) > 1:
        y_field = numeric_columns[1]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=filtered_df[numeric_field], y=filtered_df[y_field])
        plt.title(f"{numeric_field} vs {y_field}")
        plt.xlabel(numeric_field)
        plt.ylabel(y_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded and inspected using `mlcroissant`, providing insight into its structure and content.
- Numeric and categorical fields were identified for initial analysis and visualization.
- Further domain-specific data curation and feature engineering can be performed for downstream ML or statistical analysis.
